In [ ]:
import cv2
import json
import os
import shutil
import albumentations as A
import numpy as np
from functools import partial
import matplotlib.pyplot as plt

In [ ]:
def apply_morphology(img, mode='erode', **kwargs):
    kernel = np.ones((2, 2), np.uint8)
    if mode == 'erode':
        return cv2.erode(img, kernel, iterations=1)
    else:
        return cv2.dilate(img, kernel, iterations=1)

In [ ]:

transform = A.Compose([
    A.ElasticTransform(
        alpha=60,
        sigma=5,
        border_mode=cv2.BORDER_CONSTANT,
        fill=(255, 255, 255),   # ← tupla RGB blanco
        p=0.9
    ),
    A.GridDistortion(
        num_steps=4,
        distort_limit=0.3,
        border_mode=cv2.BORDER_CONSTANT,
        fill=(255, 255, 255),   # ← tupla RGB blanco
        p=0.5
    ),
    A.OneOf([
        A.Lambda(name="Erode",  image=partial(apply_morphology, mode='erode'),  p=0.5),
        A.Lambda(name="Dilate", image=partial(apply_morphology, mode='dilate'), p=0.5),
    ], p=0.7),
])

In [ ]:
def crop_borders(img, pixels=3):
    """Elimina los bordes que pueden quedar sucios tras las transformaciones."""
    h, w = img.shape[:2]
    return img[pixels:h-pixels, pixels:w-pixels]

In [ ]:
def augment_dataset(images_path, json_path, output_path, num_augments=2):
    # num_augments: 2 es suficiente para ×3 total; 4 solo si quieres ×5

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    os.makedirs(output_path, exist_ok=True)
    new_data = {}

    for img_name, text in data.items():
        img_path = os.path.join(images_path, img_name)
        image    = cv2.imread(img_path, cv2.IMREAD_COLOR_BGR)  # gris directamente

        if image is None:
            print(f"✗ No encontrada: {img_path}")
            continue

        shutil.copy(img_path, os.path.join(output_path, img_name))
        new_data[img_name] = text

        for i in range(num_augments):
            augmented = transform(image=image)["image"]
            # augmented = crop_borders(augmented, pixels=3)
            new_name  = f"aug_{i}_{img_name}"
            cv2.imwrite(os.path.join(output_path, new_name), augmented)
            new_data[new_name] = text

    with open(os.path.join(output_path, 'z_labels.json'), 'w', encoding='utf-8') as f:
        json.dump(new_data, f, indent=4, ensure_ascii=False)

    print(f"\n✅ Total imágenes: {len(new_data)} → {output_path}")

In [ ]:
img_test  = cv2.imread('IAM_img/train/img_000363.jpg')
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax in axes.flat:
    aug = transform(image=img_test)["image"]
    # aug = crop_borders(aug, pixels=3)
    ax.imshow(cv2.cvtColor(aug, cv2.COLOR_BGR2GRAY), cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
augment_dataset('IAM_img/train/', 'IAM_img/train/transcriptions.json', 'IAM_img/train_augmented/', num_augments=3)

In [ ]:
augment_dataset('final_real_dataset/train', 'final_real_dataset/train/train.json', 'final_real_dataset/augmented/', num_augments=5)

In [ ]:
augment_dataset('final_real_dataset/aparte/', 'final_real_dataset/aparte/train.json', 'final_real_dataset/aparte/augmented/', num_augments=5)

In [ ]:
augment_dataset('FINAL/final/', 'FINAL/split_final/train.json', 'FINAL/aug/', num_augments=5)

In [ ]:
def visualiza_aumentacion(ruta_imagen):
    img = cv2.imread(ruta_imagen)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    axes[0].imshow(img)
    axes[0].set_title("Fuente Original")
    axes[0].axis('off')
    
    for i in range(1, 5):
        aug = transform(image=img)["image"]
        axes[i].imshow(aug)
        axes[i].set_title(f"Variación {i}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
visualiza_aumentacion("syntetic_dataset/original/paragraph_2a51871a_2.png")

In [ ]:
augment_dataset('final_dataset','final_dataset/train.json', 'final_dataset/augmented', num_augments=4)